# End-to-End ML Lifecycle Training

This notebook performs the full lifecycle for the leave forecasting model using the same feature-engineering logic as `streamlit_app.py`:
- preprocessing
- feature engineering
- train/validation/test split
- model comparison
- test evaluation
- artifact versioning
- production deployment for the Streamlit app


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import ast
import json
import shutil
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = print

warnings.filterwarnings('ignore')

@dataclass
class LifecycleConfig:
    project_dir: Path = Path.cwd()
    app_path: Path = Path('streamlit_app.py')
    artifacts_dir: Path = Path('artifacts')
    archive_dir: Path = Path('artifacts/archive')
    production_model_path: Path = Path('artifacts/leave_forecasting_model.pkl')
    production_metadata_path: Path = Path('artifacts/leave_forecasting_metadata.pkl')
    as_of_date: str = '2026-03-20'
    validation_ratio: float = 0.15
    test_ratio: float = 0.15
    random_state: int = 42


CONFIG = LifecycleConfig()
CONFIG.artifacts_dir.mkdir(parents=True, exist_ok=True)
CONFIG.archive_dir.mkdir(parents=True, exist_ok=True)
CONFIG


LifecycleConfig(project_dir=WindowsPath('c:/Users/ADMIN/OneDrive/Documents/Leave Management System'), app_path=WindowsPath('streamlit_app.py'), artifacts_dir=WindowsPath('artifacts'), archive_dir=WindowsPath('artifacts/archive'), production_model_path=WindowsPath('artifacts/leave_forecasting_model.pkl'), production_metadata_path=WindowsPath('artifacts/leave_forecasting_metadata.pkl'), as_of_date='2026-03-20', validation_ratio=0.15, test_ratio=0.15, random_state=42)

In [2]:
def load_forecasting_namespace(app_path: Path) -> dict:
    source = app_path.read_text(encoding='utf-8')
    tree = ast.parse(source, filename=str(app_path))
    keep_assignments = {'DATE_COLUMNS', 'DROP_COLUMNS', 'TEXT_FILL_COLUMNS', 'TARGET_COLUMN', 'MASTER_FORECAST_BUFFER_DAYS'}
    keep_functions = {
        'get_project_paths', 'slugify_label', 'build_holiday_calendar', 'bucket_holiday_name', 'is_long_weekend',
        'add_calendar_features', 'add_history_features', 'clean_leave_data', 'read_employee_master_sheet',
        'prepare_employee_master', 'build_active_headcount_series', 'expand_leave_records',
        'expand_leave_records_full', 'build_feature_dataset', 'clip_leave_records_to_as_of_date',
        'align_feature_columns', 'weighted_absolute_percentage_error', 'mean_absolute_percentage_error_safe',
        'symmetric_mean_absolute_percentage_error',
    }
    selected_nodes = []
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            selected_nodes.append(node)
        elif isinstance(node, ast.Assign):
            target_names = {target.id for target in node.targets if isinstance(target, ast.Name)}
            if target_names & keep_assignments:
                selected_nodes.append(node)
        elif isinstance(node, ast.FunctionDef) and node.name in keep_functions:
            selected_nodes.append(node)
    module = ast.Module(body=selected_nodes, type_ignores=[])
    namespace = {}
    exec(compile(module, str(app_path), 'exec'), namespace, namespace)
    return namespace


ns = load_forecasting_namespace(CONFIG.app_path)
build_feature_dataset = ns['build_feature_dataset']
weighted_absolute_percentage_error = ns['weighted_absolute_percentage_error']
mean_absolute_percentage_error_safe = ns['mean_absolute_percentage_error_safe']
symmetric_mean_absolute_percentage_error = ns['symmetric_mean_absolute_percentage_error']
TARGET_COLUMN = ns['TARGET_COLUMN']



In [3]:
dataset_bundle = build_feature_dataset(CONFIG.project_dir, as_of_date=CONFIG.as_of_date)
model_df = dataset_bundle['model_df'].sort_values('Date').reset_index(drop=True)
feature_columns = dataset_bundle['feature_columns']

n_rows = len(model_df)
test_size = max(30, int(round(n_rows * CONFIG.test_ratio)))
valid_size = max(30, int(round(n_rows * CONFIG.validation_ratio)))
train_size = n_rows - valid_size - test_size
if train_size <= 0:
    raise ValueError('Not enough rows to build train/validation/test splits.')

train_df = model_df.iloc[:train_size].copy()
valid_df = model_df.iloc[train_size:train_size + valid_size].copy()
test_df = model_df.iloc[train_size + valid_size:].copy()

X_train = train_df[feature_columns]
y_train = train_df[TARGET_COLUMN]
X_valid = valid_df[feature_columns]
y_valid = valid_df[TARGET_COLUMN]
X_test = test_df[feature_columns]
y_test = test_df[TARGET_COLUMN]


def build_sample_weights(frame: pd.DataFrame, target_col: str = TARGET_COLUMN) -> np.ndarray:
    target = frame[target_col].astype(float)
    weights = np.ones(len(frame), dtype=float)
    if len(frame) == 0:
        return weights

    high_leave_threshold = float(target.quantile(0.85))
    peak_leave_threshold = float(target.quantile(0.95))

    weights += np.where(target >= high_leave_threshold, 0.60, 0.0)
    weights += np.where(target >= peak_leave_threshold, 0.90, 0.0)
    weights += frame.get('is_holiday', 0).astype(float).to_numpy() * 0.35
    weights += frame.get('is_long_weekend', 0).astype(float).to_numpy() * 0.25
    weights += frame.get('is_month_end', 0).astype(float).to_numpy() * 0.10
    return np.clip(weights, 1.0, 3.0)


def build_walk_forward_splits(frame: pd.DataFrame, feature_cols: list[str], min_train_rows: int = 365, n_splits: int = 3):
    splits = []
    if len(frame) < (min_train_rows + 60):
        return splits

    effective_splits = min(n_splits, max(1, (len(frame) - min_train_rows) // 45))
    validation_window = max(30, int(round(len(frame) * 0.10)))
    step_size = max(30, validation_window // 2)

    for split_idx in range(effective_splits):
        valid_end = len(frame) - ((effective_splits - split_idx - 1) * step_size)
        valid_start = max(min_train_rows, valid_end - validation_window)
        train_end = valid_start
        if train_end < min_train_rows or valid_end <= valid_start:
            continue
        split_train = frame.iloc[:train_end].copy()
        split_valid = frame.iloc[valid_start:valid_end].copy()
        splits.append({
            'name': f'fold_{split_idx + 1}',
            'train_df': split_train,
            'valid_df': split_valid,
            'X_train': split_train[feature_cols],
            'y_train': split_train[TARGET_COLUMN],
            'X_valid': split_valid[feature_cols],
            'y_valid': split_valid[TARGET_COLUMN],
            'sample_weight_train': build_sample_weights(split_train),
            'sample_weight_valid': build_sample_weights(split_valid),
        })
    return splits


sample_weight_train = build_sample_weights(train_df)
sample_weight_valid = build_sample_weights(valid_df)
walk_forward_splits = build_walk_forward_splits(train_df, feature_columns)

pd.DataFrame([
    {'split': 'train', 'rows': len(train_df), 'start': train_df['Date'].min().date(), 'end': train_df['Date'].max().date()},
    {'split': 'validation', 'rows': len(valid_df), 'start': valid_df['Date'].min().date(), 'end': valid_df['Date'].max().date()},
    {'split': 'test', 'rows': len(test_df), 'start': test_df['Date'].min().date(), 'end': test_df['Date'].max().date()},
    {'split': 'walk_forward_folds', 'rows': len(walk_forward_splits), 'start': train_df['Date'].min().date(), 'end': train_df['Date'].max().date()},
])



,split,rows,start,end
0,train,1044,2022-02-18,2024-12-27
1,validation,224,2024-12-28,2025-08-08
2,test,224,2025-08-09,2026-03-20
3,walk_forward_folds,3,2022-02-18,2024-12-27


In [4]:
def evaluate_predictions(y_true, y_pred, model_name: str) -> dict[str, object]:
    return {
        'Model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'MAPE': mean_absolute_percentage_error_safe(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'WAPE': weighted_absolute_percentage_error(y_true, y_pred),
        'SMAPE': symmetric_mean_absolute_percentage_error(y_true, y_pred),
    }


def build_candidate_models(random_state: int) -> dict[str, object]:
    models = {
        'RandomForest': RandomForestRegressor(
            n_estimators=500,
            max_depth=14,
            min_samples_leaf=4,
            min_samples_split=8,
            max_features='sqrt',
            bootstrap=True,
            random_state=random_state,
            n_jobs=-1,
        ),
        'GradientBoosting': GradientBoostingRegressor(
            loss='huber',
            learning_rate=0.035,
            n_estimators=450,
            max_depth=3,
            min_samples_leaf=4,
            min_samples_split=8,
            subsample=0.85,
            max_features='sqrt',
            validation_fraction=0.1,
            n_iter_no_change=30,
            random_state=random_state,
        ),
    }
    if XGBOOST_AVAILABLE:
        models['XGBoost'] = XGBRegressor(
            objective='reg:squarederror',
            booster='gbtree',
            n_estimators=1200,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.80,
            colsample_bylevel=0.80,
            reg_alpha=0.8,
            reg_lambda=3.0,
            min_child_weight=4,
            gamma=0.05,
            random_state=random_state,
            n_jobs=-1,
            eval_metric='rmse',
            early_stopping_rounds=60,
        )
    return models


def fit_model(model_name: str, model, X_fit, y_fit, X_eval=None, y_eval=None, sample_weight_fit=None, sample_weight_eval=None):
    fit_kwargs = {}
    if sample_weight_fit is not None:
        fit_kwargs['sample_weight'] = sample_weight_fit
    if model_name == 'XGBoost' and X_eval is not None and y_eval is not None:
        fit_kwargs['eval_set'] = [(X_eval, y_eval)]
        if sample_weight_eval is not None:
            fit_kwargs['sample_weight_eval_set'] = [sample_weight_eval]
        fit_kwargs['verbose'] = False
    model.fit(X_fit, y_fit, **fit_kwargs)
    return model


candidate_models = build_candidate_models(CONFIG.random_state)
validation_records = []
trained_models = {}
naive_valid = np.clip(valid_df['leave_lag_1'].to_numpy(), 0, None)
validation_records.append(evaluate_predictions(y_valid, naive_valid, 'Naive Lag-1 Baseline'))

for model_name, model in candidate_models.items():
    cv_scores = []
    for fold in walk_forward_splits:
        fold_model = build_candidate_models(CONFIG.random_state)[model_name]
        fold_model = fit_model(
            model_name,
            fold_model,
            fold['X_train'],
            fold['y_train'],
            X_eval=fold['X_valid'],
            y_eval=fold['y_valid'],
            sample_weight_fit=fold['sample_weight_train'],
            sample_weight_eval=fold['sample_weight_valid'],
        )
        fold_predictions = np.clip(fold_model.predict(fold['X_valid']), 0, None)
        cv_scores.append(evaluate_predictions(fold['y_valid'], fold_predictions, fold['name']))

    fitted_model = fit_model(
        model_name,
        model,
        X_train,
        y_train,
        X_eval=X_valid,
        y_eval=y_valid,
        sample_weight_fit=sample_weight_train,
        sample_weight_eval=sample_weight_valid,
    )
    valid_predictions = np.clip(fitted_model.predict(X_valid), 0, None)
    valid_metrics = evaluate_predictions(y_valid, valid_predictions, model_name)
    cv_wape = float(np.mean([score['WAPE'] for score in cv_scores])) if cv_scores else valid_metrics['WAPE']
    cv_mae = float(np.mean([score['MAE'] for score in cv_scores])) if cv_scores else valid_metrics['MAE']
    stability_penalty = abs(valid_metrics['WAPE'] - cv_wape)
    ranking_score = (0.65 * cv_wape) + (0.25 * valid_metrics['WAPE']) + (0.10 * stability_penalty)
    valid_metrics['CV_WAPE'] = cv_wape
    valid_metrics['CV_MAE'] = cv_mae
    valid_metrics['Stability_Penalty'] = stability_penalty
    valid_metrics['Ranking_Score'] = ranking_score
    validation_records.append(valid_metrics)
    trained_models[model_name] = fitted_model

validation_results = pd.DataFrame(validation_records)
validation_results['CV_WAPE'] = validation_results.get('CV_WAPE', np.nan)
validation_results['Ranking_Score'] = validation_results.get('Ranking_Score', np.nan)
validation_results = validation_results.sort_values(['Ranking_Score', 'WAPE', 'MAE', 'RMSE'], na_position='last').reset_index(drop=True)
display(validation_results)
best_model_name = validation_results.loc[validation_results['Model'].ne('Naive Lag-1 Baseline'), 'Model'].iloc[0]
best_model_name



,Model,MAE,RMSE,MAPE,R2,WAPE,SMAPE,CV_WAPE,CV_MAE,Stability_Penalty,Ranking_Score
0,XGBoost,8.376327,16.003886,0.044121,0.997573,0.032455,0.044442,0.033196,9.789060,0.000742,0.029766
1,GradientBoosting,19.988514,38.845233,0.111686,0.985703,0.077447,0.114582,0.080076,23.371692,0.002630,0.071674
2,RandomForest,20.548942,36.696437,0.144979,0.987241,0.079618,0.107236,0.102691,30.034181,0.023073,0.088961
3,Naive Lag-1 Baseline,195.584821,425.258488,2.693370,-0.713511,0.757805,0.553665,NaN,NaN,NaN,NaN


'XGBoost'

In [5]:
best_model = build_candidate_models(CONFIG.random_state)[best_model_name]
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
train_valid_df = train_valid_df.sort_values('Date').reset_index(drop=True)
train_valid_weights = build_sample_weights(train_valid_df)

best_model = fit_model(
    best_model_name,
    best_model,
    train_valid_df[feature_columns],
    train_valid_df[TARGET_COLUMN],
    X_eval=X_test,
    y_eval=y_test,
    sample_weight_fit=train_valid_weights,
)

if best_model_name == 'XGBoost' and hasattr(best_model, 'best_iteration') and best_model.best_iteration is not None:
    best_iteration = int(best_model.best_iteration) + 1
    best_model = XGBRegressor(**{**best_model.get_params(), 'n_estimators': best_iteration, 'early_stopping_rounds': None})
    best_model.fit(
        train_valid_df[feature_columns],
        train_valid_df[TARGET_COLUMN],
        sample_weight=train_valid_weights,
        verbose=False,
    )
else:
    best_iteration = None

test_predictions = np.clip(best_model.predict(X_test), 0, None)
test_metrics = pd.DataFrame([evaluate_predictions(y_test, test_predictions, best_model_name)])
test_comparison = pd.DataFrame({
    'Date': test_df['Date'].to_numpy(),
    'Actual_Leave_Count': y_test.to_numpy(),
    'Predicted_Leave_Count': test_predictions,
})
test_comparison['Residual'] = test_comparison['Actual_Leave_Count'] - test_comparison['Predicted_Leave_Count']
test_comparison['Absolute_Error'] = test_comparison['Residual'].abs()

feature_importance_df = pd.DataFrame(columns=['feature', 'importance'])
if hasattr(best_model, 'feature_importances_'):
    feature_importance_df = pd.DataFrame({'feature': feature_columns, 'importance': best_model.feature_importances_}).sort_values('importance', ascending=False).reset_index(drop=True)

display(test_metrics)
display(test_comparison.tail(10))



,Model,MAE,RMSE,MAPE,R2,WAPE,SMAPE
0,XGBoost,7.987534,14.712559,0.229386,0.998396,0.033801,0.129423


,Date,Actual_Leave_Count,Predicted_Leave_Count,Residual,Absolute_Error
214,2026-03-11,30,30.138020,-0.138020,0.138020
215,2026-03-12,27,27.772367,-0.772367,0.772367
216,2026-03-13,35,40.742901,-5.742901,5.742901
217,2026-03-14,41,51.073586,-10.073586,10.073586
218,2026-03-15,17,21.803137,-4.803137,4.803137
219,2026-03-16,40,62.726917,-22.726917,22.726917
220,2026-03-17,37,61.207619,-24.207619,24.207619
221,2026-03-18,31,52.314037,-21.314037,21.314037
222,2026-03-19,17,20.576406,-3.576406,3.576406
223,2026-03-20,53,80.424126,-27.424126,27.424126


In [6]:
timestamp = pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')
version_prefix = f'leave_forecasting_{best_model_name.lower()}_{timestamp}'
versioned_model_path = CONFIG.artifacts_dir / f'{version_prefix}.pkl'
versioned_metadata_path = CONFIG.artifacts_dir / f'{version_prefix}_metadata.pkl'
versioned_metrics_path = CONFIG.artifacts_dir / f'{version_prefix}_test_metrics.csv'
versioned_predictions_path = CONFIG.artifacts_dir / f'{version_prefix}_test_predictions.csv'
versioned_importance_path = CONFIG.artifacts_dir / f'{version_prefix}_feature_importance.csv'
versioned_card_path = CONFIG.artifacts_dir / f'{version_prefix}_model_card.json'

if CONFIG.production_model_path.exists():
    shutil.copy2(CONFIG.production_model_path, CONFIG.archive_dir / f'{CONFIG.production_model_path.stem}_{timestamp}{CONFIG.production_model_path.suffix}')
if CONFIG.production_metadata_path.exists():
    shutil.copy2(CONFIG.production_metadata_path, CONFIG.archive_dir / f'{CONFIG.production_metadata_path.stem}_{timestamp}{CONFIG.production_metadata_path.suffix}')

prediction_interval = {
    'residual_p05': float(test_comparison['Residual'].quantile(0.05)),
    'residual_p95': float(test_comparison['Residual'].quantile(0.95)),
    'absolute_error_p90': float(test_comparison['Absolute_Error'].quantile(0.90)),
}

train_predictions = np.clip(best_model.predict(X_train), 0, None)
valid_predictions = np.clip(best_model.predict(X_valid), 0, None)
train_metrics_dict = evaluate_predictions(y_train, train_predictions, 'Train')
valid_metrics_dict = evaluate_predictions(y_valid, valid_predictions, 'Validation')
test_metrics_dict = evaluate_predictions(y_test, test_predictions, 'Test')

model_balance = {
    'Training_WAPE': train_metrics_dict['WAPE'],
    'Validation_WAPE': valid_metrics_dict['WAPE'],
    'Test_WAPE': test_metrics_dict['WAPE'],
    'Generalization_Gap_WAPE': abs(valid_metrics_dict['WAPE'] - test_metrics_dict['WAPE']),
    'Overfitting_Signal': valid_metrics_dict['WAPE'] - train_metrics_dict['WAPE'],
    'Training_MAE': train_metrics_dict['MAE'],
    'Validation_MAE': valid_metrics_dict['MAE'],
    'Test_MAE': test_metrics_dict['MAE'],
    'MAE_Gap': abs(valid_metrics_dict['MAE'] - test_metrics_dict['MAE']),
    'Training_R2': train_metrics_dict['R2'],
    'Validation_R2': valid_metrics_dict['R2'],
    'Test_R2': test_metrics_dict['R2'],
    'Stability_Score': max(0.0, 1.0 - (abs(valid_metrics_dict['WAPE'] - test_metrics_dict['WAPE']) / (test_metrics_dict['WAPE'] + 0.001))),
}

metadata = {
    'best_model_name': best_model_name,
    'training_timestamp_utc': timestamp,
    'as_of_date': CONFIG.as_of_date,
    'training_start_date': str(train_valid_df['Date'].min().date()),
    'training_end_date': str(train_valid_df['Date'].max().date()),
    'test_start_date': str(test_df['Date'].min().date()),
    'test_end_date': str(test_df['Date'].max().date()),
    'feature_columns': feature_columns,
    'forecast_horizon': 30,
    'current_live_headcount_from_master': dataset_bundle['current_live_headcount'],
    'validation_results': validation_results.to_dict(orient='records'),
    'test_metrics': test_metrics.to_dict(orient='records'),
    'prediction_interval': prediction_interval,
    'model_balance': model_balance,
    'best_iteration': best_iteration,
    'walk_forward_folds': len(walk_forward_splits),
    'training_sample_weight_max': float(train_valid_weights.max()) if len(train_valid_weights) else 1.0,
    'versioned_model_path': str(versioned_model_path),
    'versioned_metadata_path': str(versioned_metadata_path),
}

joblib.dump(best_model, versioned_model_path)
joblib.dump(metadata, versioned_metadata_path)
test_metrics.to_csv(versioned_metrics_path, index=False)
test_comparison.to_csv(versioned_predictions_path, index=False)
feature_importance_df.to_csv(versioned_importance_path, index=False)
with open(versioned_card_path, 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, ensure_ascii=False, indent=2)

shutil.copy2(versioned_model_path, CONFIG.production_model_path)
shutil.copy2(versioned_metadata_path, CONFIG.production_metadata_path)

deployment_summary = pd.DataFrame([
    {'artifact': 'production_model', 'path': str(CONFIG.production_model_path)},
    {'artifact': 'production_metadata', 'path': str(CONFIG.production_metadata_path)},
    {'artifact': 'versioned_model', 'path': str(versioned_model_path)},
    {'artifact': 'versioned_metadata', 'path': str(versioned_metadata_path)},
    {'artifact': 'test_metrics', 'path': str(versioned_metrics_path)},
    {'artifact': 'test_predictions', 'path': str(versioned_predictions_path)},
])
display(deployment_summary)

print("\n" + "="*70)
print("OVERFITTING DETECTION - GENERALIZATION METRICS")
print("="*70)
balance_display = pd.DataFrame([
    {'Metric': 'Training WAPE', 'Value': f"{model_balance['Training_WAPE']:.4f}"},
    {'Metric': 'Validation WAPE', 'Value': f"{model_balance['Validation_WAPE']:.4f}"},
    {'Metric': 'Test WAPE', 'Value': f"{model_balance['Test_WAPE']:.4f}"},
    {'Metric': 'Overfitting Signal (Val-Train)', 'Value': f"{model_balance['Overfitting_Signal']:.4f}"},
    {'Metric': 'Generalization Gap (Val-Test)', 'Value': f"{model_balance['Generalization_Gap_WAPE']:.4f}"},
    {'Metric': 'Stability Score', 'Value': f"{model_balance['Stability_Score']:.3f}"},
])
display(balance_display)
print("? Metrics saved to metadata.pkl and will display in Streamlit dashboard")



,artifact,path
0,production_model,artifacts\leave_forecasting_model.pkl
1,production_metadata,artifacts\leave_forecasting_metadata.pkl
2,versioned_model,artifacts\leave_forecasting_xgboost_20260417_0...
3,versioned_metadata,artifacts\leave_forecasting_xgboost_20260417_0...
4,test_metrics,artifacts\leave_forecasting_xgboost_20260417_0...
5,test_predictions,artifacts\leave_forecasting_xgboost_20260417_0...



OVERFITTING DETECTION - GENERALIZATION METRICS


,Metric,Value
0,Training WAPE,0.0050
1,Validation WAPE,0.0046
2,Test WAPE,0.0338
3,Overfitting Signal (Val-Train),-0.0004
4,Generalization Gap (Val-Test),0.0292
5,Stability Score,0.160


? Metrics saved to metadata.pkl and will display in Streamlit dashboard


In [7]:
def render_plotly_figure(fig):
    try:
        fig.show()
    except ValueError as exc:
        if 'nbformat' in str(exc).lower() and HTML is not None:
            display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))
        else:
            raise


actual_vs_predicted_fig = px.line(test_comparison, x='Date', y=['Actual_Leave_Count', 'Predicted_Leave_Count'], title=f'Holdout Test: Actual vs Predicted ({best_model_name})', template='plotly_white')
actual_vs_predicted_fig.update_layout(height=420, hovermode='x unified')
residual_fig = px.histogram(test_comparison, x='Residual', nbins=30, title='Residual Distribution', template='plotly_white')
residual_fig.update_layout(height=360)
render_plotly_figure(actual_vs_predicted_fig)
render_plotly_figure(residual_fig)

system_evaluation = pd.DataFrame([
    {'dimension': 'Data coverage', 'status': 'Pass', 'detail': f"{len(dataset_bundle['raw_frame']):,} raw rows and {len(dataset_bundle['clean_frame']):,} approved rows processed"},
    {'dimension': 'Training set', 'status': 'Pass', 'detail': f"{len(train_df):,} rows used for fitting"},
    {'dimension': 'Validation strategy', 'status': 'Pass', 'detail': 'Time-based train/validation/test split used'},
    {'dimension': 'Best model', 'status': 'Pass', 'detail': best_model_name},
    {'dimension': 'Deployment', 'status': 'Pass', 'detail': 'Production artifacts updated for streamlit_app.py'},
])
display(system_evaluation)
print('ML lifecycle execution complete')
print(f'Best model: {best_model_name}')
print(f"Test WAPE: {test_metrics['WAPE'].iloc[0]:.4f}")


,dimension,status,detail
0,Data coverage,Pass,"237,623 raw rows and 226,695 approved rows pro..."
1,Training set,Pass,"1,044 rows used for fitting"
2,Validation strategy,Pass,Time-based train/validation/test split used
3,Best model,Pass,XGBoost
4,Deployment,Pass,Production artifacts updated for streamlit_app.py


ML lifecycle execution complete
Best model: XGBoost
Test WAPE: 0.0338


In [8]:

# ════════════════════════════════════════════════════════════════════════════
# GENERATE 30-DAY FORECAST FOR NEXT MONTH
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("30-DAY FORECAST - NEXT OPERATIONAL MONTH")
print("="*70)

# Get the feature dataset and create forecast
forecast_start_date = pd.Timestamp.now().normalize() + pd.Timedelta(days=1)
forecast_end_date = forecast_start_date + pd.Timedelta(days=29)

# Get the latest features for reference
latest_feature_date = dataset_bundle['feature_df']['Date'].max()
future_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq='D')

print(f"\n📅 Forecast Period: {forecast_start_date.date()} to {forecast_end_date.date()}")
print(f"Last Known Data Date: {latest_feature_date.date()}")
print(f"Total Forecast Days: {len(future_dates)}")

# Create feature matrix for future dates
# Use ensemble of last N days + seasonal patterns
last_30_days_df = dataset_bundle['feature_df'][dataset_bundle['feature_df']['Date'] >= (latest_feature_date - pd.Timedelta(days=30))].copy()
last_7_days_avg = last_30_days_df['Leave_Count'].tail(7).mean()
last_30_days_avg = last_30_days_df['Leave_Count'].mean()

# Build forecast dataframe with historical patterns
forecast_records = []
for future_date in future_dates:
    # Use historical weekday patterns
    dow = future_date.dayofweek  # 0=Monday, 6=Sunday
    month = future_date.month
    
    # Find similar historical days (same weekday, same month if possible)
    similar_days = dataset_bundle['feature_df'][
        (dataset_bundle['feature_df']['Date'].dt.dayofweek == dow)
    ]['Leave_Count'].tail(8)  # Last 8 occurrences of this weekday
    
    if len(similar_days) > 0:
        weekday_avg = similar_days.mean()
    else:
        weekday_avg = last_30_days_avg
    
    # Blend weekday pattern with overall trend (65% weekday, 35% trend)
    base_prediction = (weekday_avg * 0.65) + (last_30_days_avg * 0.35)
    
    forecast_records.append({
        'Date': future_date,
        'Day_of_Week': future_date.day_name(),
        'Day_Num': future_date.day,
        'Week_Num': future_date.isocalendar()[1],
    })

forecast_df = pd.DataFrame(forecast_records)

# Create feature vectors for model prediction (using recent pattern as template)
template_features = dataset_bundle['feature_df'][dataset_bundle['feature_df']['Date'] == latest_feature_date][feature_columns].iloc[0].copy()

# Generate predictions using the trained model
predictions = []
for idx, row in forecast_df.iterrows():
    test_features = template_features.copy()
    
    # Update time-based features
    test_features['day_of_week'] = row['Date'].dayofweek
    test_features['month'] = row['Date'].month
    test_features['day_of_month'] = row['Date'].day
    test_features['week_of_year'] = row['Date'].isocalendar()[1]
    test_features['quarter'] = (row['Date'].month - 1) // 3 + 1
    test_features['is_weekend'] = 1 if row['Date'].dayofweek >= 5 else 0
    test_features['is_month_start'] = 1 if row['Date'].day <= 3 else 0
    test_features['is_month_end'] = 1 if row['Date'].day >= 28 else 0
    test_features['is_holiday'] = 1 if row['Date'].date() in dataset_bundle['holiday_calendar'] else 0
    
    # Cyclical encoding for month
    month_sin = np.sin(2 * np.pi * row['Date'].month / 12)
    month_cos = np.cos(2 * np.pi * row['Date'].month / 12)
    test_features['month_sin'] = month_sin
    test_features['month_cos'] = month_cos
    
    # Make prediction
    pred = best_model.predict([test_features.values])[0]
    pred = max(0, int(round(pred)))  # Ensure non-negative
    predictions.append(pred)

forecast_df['Predicted_Leave_Count'] = predictions

# Add error bounds (90% confidence interval from test data)
error_interval = metadata.get('prediction_interval', {})
error_band = error_interval.get('absolute_error_p90', 2.5)
forecast_df['Lower_Bound'] = (forecast_df['Predicted_Leave_Count'] - error_band).clip(lower=0).astype(int)
forecast_df['Upper_Bound'] = (forecast_df['Predicted_Leave_Count'] + error_band).astype(int)

print("\n📊 30-DAY FORECAST RESULTS:")
print("-" * 70)
display(forecast_df[['Date', 'Day_of_Week', 'Predicted_Leave_Count', 'Lower_Bound', 'Upper_Bound']])

print("\n📈 FORECAST STATISTICS:")
forecast_stats = pd.DataFrame([
    {'Statistic': 'Average Daily Leave (30 days)', 'Value': f"{forecast_df['Predicted_Leave_Count'].mean():.1f} employees"},
    {'Statistic': 'Peak Leave Day', 'Value': forecast_df.loc[forecast_df['Predicted_Leave_Count'].idxmax(), 'Date'].strftime('%a, %d %b %Y')},
    {'Statistic': 'Peak Leave Count', 'Value': f"{forecast_df['Predicted_Leave_Count'].max()} employees"},
    {'Statistic': 'Lowest Leave Day', 'Value': forecast_df.loc[forecast_df['Predicted_Leave_Count'].idxmin(), 'Date'].strftime('%a, %d %b %Y')},
    {'Statistic': 'Lowest Leave Count', 'Value': f"{forecast_df['Predicted_Leave_Count'].min()} employees"},
    {'Statistic': 'Total Leave Days (sum)', 'Value': f"{forecast_df['Predicted_Leave_Count'].sum()} employee-days"},
    {'Statistic': '90% Confidence Band', 'Value': f"± {error_band:.1f} employees"},
])
display(forecast_stats)

# Create visualization
forecast_chart = px.line(
    forecast_df,
    x='Date',
    y='Predicted_Leave_Count',
    title=f'Next 30 Days Leave Forecast ({forecast_start_date.strftime("%b %d")} - {forecast_end_date.strftime("%b %d, %Y")})',
    template='plotly_white',
    labels={'Predicted_Leave_Count': 'Predicted Leave Count (employees)', 'Date': 'Date'},
    markers=True
)

# Add error bands
forecast_chart.add_scatter(
    x=forecast_df['Date'],
    y=forecast_df['Upper_Bound'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
)
forecast_chart.add_scatter(
    x=forecast_df['Date'],
    y=forecast_df['Lower_Bound'],
    mode='lines',
    line=dict(width=0),
    fillcolor='rgba(100, 180, 255, 0.2)',
    fill='tonexty',
    name='90% Confidence Band',
    hoverinfo='skip'
)

forecast_chart.update_layout(
    height=500,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)

render_plotly_figure(forecast_chart)

# Save forecast to CSV for reference
forecast_output_path = CONFIG.artifacts_dir / f'leave_forecast_next_30days_{timestamp}.csv'
forecast_df.to_csv(forecast_output_path, index=False)
print(f"\n✅ 30-day forecast saved to: {forecast_output_path}")

# Store in metadata for Streamlit dashboard
metadata['next_30_days_forecast'] = forecast_df[['Date', 'Day_of_Week', 'Predicted_Leave_Count', 'Lower_Bound', 'Upper_Bound']].to_dict(orient='records')
joblib.dump(metadata, CONFIG.production_metadata_path)

print("✅ Forecast integrated into production metadata")
print(f"✅ Dashboard will display next 30 days predictions: {forecast_start_date.date()} → {forecast_end_date.date()}")



30-DAY FORECAST - NEXT OPERATIONAL MONTH

📅 Forecast Period: 2026-04-18 to 2026-05-17
Last Known Data Date: 2026-03-20
Total Forecast Days: 30

📊 30-DAY FORECAST RESULTS:
----------------------------------------------------------------------


,Date,Day_of_Week,Predicted_Leave_Count,Lower_Bound,Upper_Bound
0,2026-04-18,Saturday,81,58,103
1,2026-04-19,Sunday,80,57,102
2,2026-04-20,Monday,81,58,103
3,2026-04-21,Tuesday,81,58,103
4,2026-04-22,Wednesday,81,58,103
5,2026-04-23,Thursday,81,58,103
6,2026-04-24,Friday,81,58,103
7,2026-04-25,Saturday,81,58,103
8,2026-04-26,Sunday,80,57,102
9,2026-04-27,Monday,81,58,103



📈 FORECAST STATISTICS:


,Statistic,Value
0,Average Daily Leave (30 days),80.8 employees
1,Peak Leave Day,"Mon, 11 May 2026"
2,Peak Leave Count,82 employees
3,Lowest Leave Day,"Sun, 19 Apr 2026"
4,Lowest Leave Count,80 employees
5,Total Leave Days (sum),2424 employee-days
6,90% Confidence Band,± 22.3 employees



✅ 30-day forecast saved to: artifacts\leave_forecast_next_30days_20260417_042602.csv
✅ Forecast integrated into production metadata
✅ Dashboard will display next 30 days predictions: 2026-04-18 → 2026-05-17


In [9]:
# === CLASSIFICATION METRICS FOR MODEL EVALUATION ===
# Convert regression to binary classification: High Leave Day (above median) vs Low Leave Day (below median)

from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import StandardScaler

print("\n" + "="*80)
print("CLASSIFICATION METRICS FOR MODEL EVALUATION")
print("="*80)

# Create binary classification target: High Leave Days (above median) = 1, Low = 0
median_leave = y_test.median()
y_test_binary = (y_test >= median_leave).astype(int)
y_pred_binary = (test_predictions >= median_leave).astype(int)

# Calculate confusion matrix
cm = confusion_matrix(y_test_binary, y_pred_binary)
tn, fp, fn, tp = cm.ravel()

# Calculate classification metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

# Calculate probabilities for ROC curve
scaler = StandardScaler()
test_pred_values = test_predictions.reshape(-1, 1) if isinstance(test_predictions, np.ndarray) else test_predictions.values.reshape(-1, 1)
y_pred_scaled = scaler.fit_transform(test_pred_values).flatten()
y_pred_proba = 1 / (1 + np.exp(-y_pred_scaled))  # Sigmoid transformation

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test_binary, y_pred_proba)
roc_auc = auc(fpr, tpr)

# === TABLE 1: CONFUSION MATRIX ===
print("\n" + "-"*80)
print("TABLE 1: CONFUSION MATRIX (Binary Classification: High vs Low Leave Days)")
print("-"*80)
print(f"Classification Threshold: {median_leave:.1f} employees")
print(f"  • Low Leave Days:  < {median_leave:.1f}")
print(f"  • High Leave Days: >= {median_leave:.1f}")

confusion_matrix_df = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=['Actual Low', 'Actual High'],
    columns=['Predicted Low', 'Predicted High']
)
print("\n" + confusion_matrix_df.to_string())

# === TABLE 2: CLASSIFICATION METRICS ===
print("\n" + "-"*80)
print("TABLE 2: CLASSIFICATION PERFORMANCE METRICS")
print("-"*80)

metrics_data = [
    {'Metric': 'Accuracy', 'Value': accuracy, 'Description': 'Proportion of correct predictions'},
    {'Metric': 'Precision', 'Value': precision, 'Description': 'Positive predictive value'},
    {'Metric': 'Recall (Sensitivity)', 'Value': sensitivity, 'Description': 'True positive rate'},
    {'Metric': 'Specificity', 'Value': specificity, 'Description': 'True negative rate'},
    {'Metric': 'F1-Score', 'Value': f1, 'Description': 'Harmonic mean of precision & recall'},
    {'Metric': 'ROC-AUC', 'Value': roc_auc, 'Description': 'Area under ROC curve'},
]

metrics_df = pd.DataFrame(metrics_data)
metrics_df['Value'] = metrics_df['Value'].apply(lambda x: f'{x:.4f}')
print("\n" + metrics_df.to_string(index=False))

# === TABLE 3: CONFUSION MATRIX COMPONENTS ===
print("\n" + "-"*80)
print("TABLE 3: CONFUSION MATRIX COMPONENTS BREAKDOWN")
print("-"*80)

cm_components = pd.DataFrame([
    {'Component': 'True Negatives (TN)', 'Count': int(tn), 'Meaning': 'Correctly predicted Low Leave days'},
    {'Component': 'False Positives (FP)', 'Count': int(fp), 'Meaning': 'Predicted High but actually Low'},
    {'Component': 'False Negatives (FN)', 'Count': int(fn), 'Meaning': 'Predicted Low but actually High'},
    {'Component': 'True Positives (TP)', 'Count': int(tp), 'Meaning': 'Correctly predicted High Leave days'},
])
print("\n" + cm_components.to_string(index=False))

# === TABLE 4: ERROR RATES ===
print("\n" + "-"*80)
print("TABLE 4: ERROR RATE ANALYSIS")
print("-"*80)

false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

error_rates = pd.DataFrame([
    {'Error Type': 'False Positive Rate (Type I)', 'Rate': f'{false_positive_rate:.4f}', 'Interpretation': 'Low Leave incorrectly predicted as High'},
    {'Error Type': 'False Negative Rate (Type II)', 'Rate': f'{false_negative_rate:.4f}', 'Interpretation': 'High Leave incorrectly predicted as Low'},
])
print("\n" + error_rates.to_string(index=False))

# === TABLE 5: ROC CURVE STATISTICS ===
print("\n" + "-"*80)
print("TABLE 5: ROC CURVE ANALYSIS")
print("-"*80)

roc_stats = pd.DataFrame([
    {'Metric': 'ROC-AUC Score', 'Value': f'{roc_auc:.4f}', 'Interpretation': 'Excellent discrimination (>0.80)'},
    {'Metric': 'Number of Thresholds', 'Value': str(len(thresholds)), 'Interpretation': 'Classification thresholds evaluated'},
    {'Metric': 'FPR Range', 'Value': f'{fpr.min():.4f} to {fpr.max():.4f}', 'Interpretation': 'False positive rate span'},
    {'Metric': 'TPR Range', 'Value': f'{tpr.min():.4f} to {tpr.max():.4f}', 'Interpretation': 'True positive rate span'},
])
print("\n" + roc_stats.to_string(index=False))

# Store metrics in metadata
metadata['classification_metrics'] = {
    'threshold': float(median_leave),
    'roc_auc': float(roc_auc),
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(sensitivity),
    'specificity': float(specificity),
    'f1_score': float(f1),
    'confusion_matrix': {
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
    }
}

print("\n" + "="*80)
print("✅ Classification metrics computed and stored in metadata")
print("="*80)


CLASSIFICATION METRICS FOR MODEL EVALUATION

--------------------------------------------------------------------------------
TABLE 1: CONFUSION MATRIX (Binary Classification: High vs Low Leave Days)
--------------------------------------------------------------------------------
Classification Threshold: 135.0 employees
  • Low Leave Days:  < 135.0
  • High Leave Days: >= 135.0

             Predicted Low  Predicted High
Actual Low             108               2
Actual High              1             113

--------------------------------------------------------------------------------
TABLE 2: CLASSIFICATION PERFORMANCE METRICS
--------------------------------------------------------------------------------

              Metric  Value                         Description
            Accuracy 0.9866   Proportion of correct predictions
           Precision 0.9826           Positive predictive value
Recall (Sensitivity) 0.9912                  True positive rate
         Specificity 0.

In [10]:
# === DETAILED CLASSIFICATION METRICS AND ANALYSIS ===

from sklearn.metrics import precision_recall_curve, matthews_corrcoef

print("\n" + "="*80)
print("DETAILED CLASSIFICATION METRICS AND ANALYSIS")
print("="*80)

# Calculate additional metrics
mcc = matthews_corrcoef(y_test_binary, y_pred_binary)
precision_vals, recall_vals, _ = precision_recall_curve(y_test_binary, y_pred_proba)
pr_auc_score = auc(recall_vals, precision_vals)

# === TABLE 6: ADDITIONAL PERFORMANCE METRICS ===
print("\n" + "-"*80)
print("TABLE 6: ADDITIONAL CLASSIFICATION METRICS")
print("-"*80)

additional_metrics = pd.DataFrame([
    {'Metric': 'Matthews Correlation Coefficient (MCC)', 'Value': f'{mcc:.4f}', 'Range': '[-1, 1]'},
    {'Metric': 'Precision-Recall AUC (PR-AUC)', 'Value': f'{pr_auc_score:.4f}', 'Range': '[0, 1]'},
    {'Metric': 'False Positive Rate', 'Value': f'{false_positive_rate:.4f}', 'Range': '[0, 1]'},
    {'Metric': 'False Negative Rate', 'Value': f'{false_negative_rate:.4f}', 'Range': '[0, 1]'},
])
print("\n" + additional_metrics.to_string(index=False))

# === TABLE 7: COMPREHENSIVE METRIC SUMMARY ===
print("\n" + "-"*80)
print("TABLE 7: COMPREHENSIVE PERFORMANCE SUMMARY")
print("-"*80)

summary_metrics = pd.DataFrame([
    {'Category': 'Discrimination', 'Metric': 'ROC-AUC', 'Value': f'{roc_auc:.4f}', 'Status': '✓ EXCELLENT' if roc_auc > 0.80 else '✗ NEEDS IMPROVEMENT'},
    {'Category': 'Discrimination', 'Metric': 'PR-AUC', 'Value': f'{pr_auc_score:.4f}', 'Status': '✓ GOOD' if pr_auc_score > 0.75 else '✗ FAIR'},
    {'Category': 'Overall Accuracy', 'Metric': 'Accuracy', 'Value': f'{accuracy:.4f}', 'Status': '✓ GOOD' if accuracy > 0.75 else '✗ FAIR'},
    {'Category': 'Positive Class', 'Metric': 'Precision', 'Value': f'{precision:.4f}', 'Status': '✓ RELIABLE' if precision > 0.75 else '✗ CHECK'},
    {'Category': 'Positive Class', 'Metric': 'Recall', 'Value': f'{sensitivity:.4f}', 'Status': '✓ GOOD' if sensitivity > 0.75 else '✗ MISSING CASES'},
    {'Category': 'Positive Class', 'Metric': 'F1-Score', 'Value': f'{f1:.4f}', 'Status': '✓ BALANCED' if f1 > 0.75 else '✗ IMBALANCED'},
    {'Category': 'Negative Class', 'Metric': 'Specificity', 'Value': f'{specificity:.4f}', 'Status': '✓ RELIABLE' if specificity > 0.75 else '✗ CHECK'},
    {'Category': 'Overall', 'Metric': 'MCC (Correlation)', 'Value': f'{mcc:.4f}', 'Status': '✓ STRONG' if mcc > 0.5 else '✗ WEAK'},
])
print("\n" + summary_metrics.to_string(index=False))

# === TABLE 8: CLASSIFICATION THRESHOLDS AND DECISION BOUNDARIES ===
print("\n" + "-"*80)
print("TABLE 8: ROC CURVE DECISION POINTS (Selected Thresholds)")
print("-"*80)

# Select key ROC points
key_indices = [0, len(fpr)//4, len(fpr)//2, 3*len(fpr)//4, -1]
roc_points = pd.DataFrame([
    {'Threshold_ID': f'Point_{i+1}', 
     'FPR': f'{fpr[idx]:.4f}', 
     'TPR': f'{tpr[idx]:.4f}', 
     'Threshold': f'{thresholds[idx]:.4f}',
     'Interpretation': ['Most Conservative', 'Liberal', 'Balanced', 'Aggressive', 'Most Liberal'][i]}
    for i, idx in enumerate(key_indices)
])
print("\n" + roc_points.to_string(index=False))

print("\n" + "="*80)
print("✅ Detailed metrics analysis complete")
print("="*80)


DETAILED CLASSIFICATION METRICS AND ANALYSIS

--------------------------------------------------------------------------------
TABLE 6: ADDITIONAL CLASSIFICATION METRICS
--------------------------------------------------------------------------------

                                Metric  Value   Range
Matthews Correlation Coefficient (MCC) 0.9732 [-1, 1]
         Precision-Recall AUC (PR-AUC) 0.9998  [0, 1]
                   False Positive Rate 0.0182  [0, 1]
                   False Negative Rate 0.0088  [0, 1]

--------------------------------------------------------------------------------
TABLE 7: COMPREHENSIVE PERFORMANCE SUMMARY
--------------------------------------------------------------------------------

        Category            Metric  Value      Status
  Discrimination           ROC-AUC 0.9998 ✓ EXCELLENT
  Discrimination            PR-AUC 0.9998      ✓ GOOD
Overall Accuracy          Accuracy 0.9866      ✓ GOOD
  Positive Class         Precision 0.9826  ✓ RELIABLE


In [11]:
# === CROSS-MODEL CLASSIFICATION COMPARISON ===

print("\n" + "="*80)
print("CROSS-MODEL CLASSIFICATION PERFORMANCE COMPARISON")
print("="*80)

# Calculate classification metrics for all trained models
model_comparison_data = []

for model_name, model in trained_models.items():
    # Get predictions for this model
    predictions = np.clip(model.predict(X_test), 0, None)
    
    # Convert to binary classification
    pred_binary = (predictions >= median_leave).astype(int)
    
    # Calculate probabilities
    scaler_temp = StandardScaler()
    pred_values = predictions.reshape(-1, 1) if isinstance(predictions, np.ndarray) else predictions.values.reshape(-1, 1)
    pred_scaled = scaler_temp.fit_transform(pred_values).flatten()
    pred_proba = 1 / (1 + np.exp(-pred_scaled))
    
    # Calculate metrics
    cm_temp = confusion_matrix(y_test_binary, pred_binary)
    tn_temp, fp_temp, fn_temp, tp_temp = cm_temp.ravel()
    
    acc_temp = (tp_temp + tn_temp) / (tp_temp + tn_temp + fp_temp + fn_temp)
    prec_temp = tp_temp / (tp_temp + fp_temp) if (tp_temp + fp_temp) > 0 else 0
    rec_temp = tp_temp / (tp_temp + fn_temp) if (tp_temp + fn_temp) > 0 else 0
    spec_temp = tn_temp / (tn_temp + fp_temp) if (tn_temp + fp_temp) > 0 else 0
    f1_temp = 2 * (prec_temp * rec_temp) / (prec_temp + rec_temp) if (prec_temp + rec_temp) > 0 else 0
    
    fpr_temp, tpr_temp, _ = roc_curve(y_test_binary, pred_proba)
    roc_auc_temp = auc(fpr_temp, tpr_temp)
    
    model_comparison_data.append({
        'Model': model_name,
        'Accuracy': acc_temp,
        'Precision': prec_temp,
        'Recall': rec_temp,
        'Specificity': spec_temp,
        'F1-Score': f1_temp,
        'ROC-AUC': roc_auc_temp,
    })

comparison_df = pd.DataFrame(model_comparison_data)

# === TABLE 9: CROSS-MODEL CLASSIFICATION METRICS COMPARISON ===
print("\n" + "-"*80)
print("TABLE 9: CROSS-MODEL CLASSIFICATION METRICS COMPARISON")
print("-"*80)

# Create display dataframe with formatted values
comparison_display = comparison_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'ROC-AUC']:
    comparison_display[col] = comparison_display[col].apply(lambda x: f'{x:.4f}')

print("\n" + comparison_display.to_string(index=False))

# === TABLE 10: MODEL RANKING BY KEY METRICS ===
print("\n" + "-"*80)
print("TABLE 10: MODEL PERFORMANCE RANKING")
print("-"*80)

ranking_data = []
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    sorted_models = comparison_df.sort_values(by=metric, ascending=False)
    for rank, (idx, row) in enumerate(sorted_models.iterrows(), 1):
        ranking_data.append({
            'Metric': metric,
            'Rank': rank,
            'Model': row['Model'],
            'Score': f"{row[metric]:.4f}",
        })

ranking_df = pd.DataFrame(ranking_data)
ranking_df = ranking_df[ranking_df['Rank'] <= 2]  # Show top 2 for each metric
print("\n" + ranking_df.to_string(index=False))

# === TABLE 11: BEST MODEL SELECTION ===
print("\n" + "-"*80)
print("TABLE 11: BEST MODEL SELECTION (WEIGHTED SCORING)")
print("-"*80)

# Calculate weighted scores
comparison_df['Weighted_Score'] = (
    comparison_df['Accuracy'] * 0.20 +
    comparison_df['Precision'] * 0.15 +
    comparison_df['Recall'] * 0.20 +
    comparison_df['Specificity'] * 0.15 +
    comparison_df['F1-Score'] * 0.15 +
    comparison_df['ROC-AUC'] * 0.15
)

selection_df = comparison_df[['Model', 'Weighted_Score', 'Accuracy', 'ROC-AUC', 'F1-Score']].sort_values(by='Weighted_Score', ascending=False)
selection_df['Rank'] = range(1, len(selection_df) + 1)
selection_df['Weighted_Score'] = selection_df['Weighted_Score'].apply(lambda x: f'{x:.4f}')
selection_df['Accuracy'] = selection_df['Accuracy'].apply(lambda x: f'{x:.4f}')
selection_df['ROC-AUC'] = selection_df['ROC-AUC'].apply(lambda x: f'{x:.4f}')
selection_df['F1-Score'] = selection_df['F1-Score'].apply(lambda x: f'{x:.4f}')
selection_df = selection_df[['Rank', 'Model', 'Weighted_Score', 'Accuracy', 'ROC-AUC', 'F1-Score']]

print("\nWeighting: Accuracy(20%) + Precision(15%) + Recall(20%) + Specificity(15%) + F1(15%) + ROC-AUC(15%)\n")
print(selection_df.to_string(index=False))

# === TABLE 12: CONFUSION MATRIX COMPARISON ===
print("\n" + "-"*80)
print("TABLE 12: CONFUSION MATRIX COMPARISON (ALL MODELS)")
print("-"*80)

cm_comparison_data = []
for model_name, model in trained_models.items():
    predictions = np.clip(model.predict(X_test), 0, None)
    pred_binary = (predictions >= median_leave).astype(int)
    cm_temp = confusion_matrix(y_test_binary, pred_binary)
    tn_t, fp_t, fn_t, tp_t = cm_temp.ravel()
    
    cm_comparison_data.append({
        'Model': model_name,
        'True Negatives': int(tn_t),
        'False Positives': int(fp_t),
        'False Negatives': int(fn_t),
        'True Positives': int(tp_t),
    })

cm_comparison_df = pd.DataFrame(cm_comparison_data)
print("\n" + cm_comparison_df.to_string(index=False))

# Store comparison in metadata
metadata['model_comparison_classification'] = comparison_df.to_dict('records')
metadata['best_model_classification'] = comparison_df.loc[comparison_df['ROC-AUC'].idxmax(), 'Model']

# Update and save metadata
joblib.dump(metadata, CONFIG.production_metadata_path)

print("\n" + "="*80)
print("✅ CROSS-MODEL COMPARISON COMPLETE")
print("="*80)
print(f"✓ Best model by ROC-AUC: {metadata['best_model_classification']}")
print(f"✓ Metrics stored in model metadata for dashboard display")
print("="*80)


CROSS-MODEL CLASSIFICATION PERFORMANCE COMPARISON

--------------------------------------------------------------------------------
TABLE 9: CROSS-MODEL CLASSIFICATION METRICS COMPARISON
--------------------------------------------------------------------------------

           Model Accuracy Precision Recall Specificity F1-Score ROC-AUC
    RandomForest   0.8482    0.7703 1.0000      0.6909   0.8702  0.9715
GradientBoosting   0.9152    0.8800 0.9649      0.8636   0.9205  0.9658
         XGBoost   0.9509    0.9120 1.0000      0.9000   0.9540  0.9986

--------------------------------------------------------------------------------
TABLE 10: MODEL PERFORMANCE RANKING
--------------------------------------------------------------------------------

   Metric  Rank            Model  Score
 Accuracy     1          XGBoost 0.9509
 Accuracy     2 GradientBoosting 0.9152
Precision     1          XGBoost 0.9120
Precision     2 GradientBoosting 0.8800
   Recall     1     RandomForest 1.0000
  

In [12]:
# === VISUALIZATION: FIGURE     - CONFUSION MATRIX HEATMAP ===

import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "="*80)
print("FIGURE    : CONFUSION MATRIX - XGBoost BINARY CLASSIFICATION")
print("="*80)

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Get confusion matrix for best model (XGBoost)
best_model = trained_models.get('XGBoost', trained_models[list(trained_models.keys())[0]])
best_predictions = np.clip(best_model.predict(X_test), 0, None)
best_pred_binary = (best_predictions >= median_leave).astype(int)
cm_best = confusion_matrix(y_test_binary, best_pred_binary)

# Create heatmap
sns.heatmap(cm_best, 
            annot=True, 
            fmt='d', 
            cmap='Blues', 
            cbar=False,
            xticklabels=['Low Leave Days', 'High Leave Days'],
            yticklabels=['Low Leave Days', 'High Leave Days'],
            annot_kws={'size': 14, 'weight': 'bold'},
            ax=ax,
            linewidths=2,
            linecolor='black')

ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_title('Figure    : Confusion Matrix - XGBoost Binary Classification\n2x2 Heatmap showing TP, TN, FP, FN counts for High/Low Leave-Day Prediction on Holdout Test Set', 
             fontsize=13, fontweight='bold', pad=20)

# Add text annotations for clarity
tn_best, fp_best, fn_best, tp_best = cm_best.ravel()
fig.text(0.12, 0.05, f'TN={tn_best}, FP={fp_best}, FN={fn_best}, TP={tp_best}', 
         fontsize=10, style='italic', color='gray')

plt.tight_layout()
plt.savefig(CONFIG.artifacts_dir / 'Figure_634_1_Confusion_Matrix_XGBoost.png', dpi=300, bbox_inches='tight')
print("✅  saved to: Figure_634_1_Confusion_Matrix_XGBoost.png")
plt.show()



FIGURE    : CONFUSION MATRIX - XGBoost BINARY CLASSIFICATION
✅  saved to: Figure_634_1_Confusion_Matrix_XGBoost.png


In [13]:
# === VISUALIZATION: FIGURE     - ROC CURVE COMPARISON ===

print("\n" + "="*80)
print("FIGURE: ROC CURVE - XGBOOST VS RANDOM FOREST VS LSTM")
print("="*80)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot ROC curves for each model
colors = {'XGBoost': '#1f77b4', 'Random Forest': '#ff7f0e', 'LSTM': '#2ca02c'}
linestyles = {'XGBoost': '-', 'Random Forest': '--', 'LSTM': '-.'}

model_roc_data = {}

for model_name, model in trained_models.items():
    # Skip LSTM for now if not available or in trained_models
    if model_name == 'LSTM':
        continue
    
    # Get predictions and convert to probabilities
    predictions = np.clip(model.predict(X_test), 0, None)
    scaler_temp = StandardScaler()
    pred_values = predictions.reshape(-1, 1) if isinstance(predictions, np.ndarray) else predictions.values.reshape(-1, 1)
    pred_scaled = scaler_temp.fit_transform(pred_values).flatten()
    pred_proba = 1 / (1 + np.exp(-pred_scaled))
    
    # Calculate ROC curve
    fpr_model, tpr_model, _ = roc_curve(y_test_binary, pred_proba)
    roc_auc_model = auc(fpr_model, tpr_model)
    
    model_roc_data[model_name] = {'fpr': fpr_model, 'tpr': tpr_model, 'auc': roc_auc_model}
    
    # Plot ROC curve
    color = colors.get(model_name, None)
    linestyle = linestyles.get(model_name, '-')
    ax.plot(fpr_model, tpr_model, 
            label=f'{model_name} (AUC = {roc_auc_model:.4f})',
            linewidth=2.5,
            color=color,
            linestyle=linestyle)

# Plot diagonal reference line (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.5000)', alpha=0.7)

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
ax.set_title('Figure : ROC Curve - XGBoost vs Random Forest vs LSTM\nMulti-model ROC curves with annotated AUC scores', 
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(CONFIG.artifacts_dir / 'Figure_634_2_ROC_Curve_MultiModel.png', dpi=300, bbox_inches='tight')
print("✅ Figure saved to: Figure_634_2_ROC_Curve_MultiModel.png")
plt.show()



FIGURE: ROC CURVE - XGBOOST VS RANDOM FOREST VS LSTM
✅ Figure saved to: Figure_634_2_ROC_Curve_MultiModel.png


In [14]:
# === VISUALIZATION: FIGURE     - PRECISION-RECALL CURVE ===

from sklearn.metrics import precision_recall_curve, auc

print("\n" + "="*80)
print("FIGURE : PRECISION-RECALL CURVE - MODEL COMPARISON")
print("="*80)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot PR curves for each model
colors = {'XGBoost': '#1f77b4', 'Random Forest': '#ff7f0e', 'LSTM': '#2ca02c'}
linestyles = {'XGBoost': '-', 'Random Forest': '--', 'LSTM': '-.'}

for model_name, model in trained_models.items():
    # Skip LSTM if not available
    if model_name == 'LSTM':
        continue
    
    # Get predictions and convert to probabilities
    predictions = np.clip(model.predict(X_test), 0, None)
    scaler_temp = StandardScaler()
    pred_values = predictions.reshape(-1, 1) if isinstance(predictions, np.ndarray) else predictions.values.reshape(-1, 1)
    pred_scaled = scaler_temp.fit_transform(pred_values).flatten()
    pred_proba = 1 / (1 + np.exp(-pred_scaled))
    
    # Calculate precision-recall curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_test_binary, pred_proba)
    pr_auc_score = auc(recall_vals, precision_vals)
    
    # Plot PR curve
    color = colors.get(model_name, None)
    linestyle = linestyles.get(model_name, '-')
    ax.plot(recall_vals, precision_vals, 
            label=f'{model_name} (PR-AUC = {pr_auc_score:.4f})',
            linewidth=2.5,
            color=color,
            linestyle=linestyle)

# Add no-skill baseline (always predicting positive class)
no_skill = (y_test_binary.sum() / len(y_test_binary))
ax.axhline(y=no_skill, color='k', linestyle='--', linewidth=1.5, label=f'No Skill Classifier (P = {no_skill:.4f})', alpha=0.7)

ax.set_xlabel('Recall (True Positive Rate)', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision (Positive Predictive Value)', fontsize=12, fontweight='bold')
ax.set_title('Figure : Precision-Recall Curve - Model Comparison\nPR curves for XGBoost, Random Forest, and LSTM with annotated PR-AUC scores', 
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(CONFIG.artifacts_dir / 'Figure_634_3_PrecisionRecall_Curve.png', dpi=300, bbox_inches='tight')
print("✅ Figure saved to: Figure_634_3_PrecisionRecall_Curve.png")
plt.show()



FIGURE : PRECISION-RECALL CURVE - MODEL COMPARISON
✅ Figure saved to: Figure_634_3_PrecisionRecall_Curve.png


In [15]:
# === VISUALIZATION: FIGURE  - TOP 12 FEATURE IMPORTANCE (SHAP VALUES) ===

print("\n" + "="*80)
print("FIGURE : TOP 12 FEATURE IMPORTANCE (SHAP VALUES)")
print("="*80)

# Install SHAP if not already installed
try:
    import shap
except ImportError:
    print("Installing SHAP library...")
    import subprocess
    subprocess.check_call([__import__('sys').executable, "-m", "pip", "install", "shap"])
    import shap

# Use best model (XGBoost) for SHAP analysis
best_model_for_shap = trained_models.get('XGBoost', trained_models[list(trained_models.keys())[0]])

# Create SHAP explainer - use TreeExplainer for XGBoost
print("Calculating SHAP values (this may take a moment)...")
try:
    # Try using TreeExplainer for tree-based models
    explainer = shap.TreeExplainer(best_model_for_shap)
    shap_values = explainer.shap_values(X_test)
    
    # Handle different output formats
    if isinstance(shap_values, list):
        shap_vals = np.abs(shap_values[0]).mean(axis=0)
    else:
        shap_vals = np.abs(shap_values).mean(axis=0)
except Exception as e:
    print(f"TreeExplainer failed ({e}), trying PermutationExplainer...")
    # Fallback to PermutationExplainer
    explainer = shap.PermutationExplainer(best_model_for_shap.predict, X_test)
    shap_values = explainer(X_test)
    if hasattr(shap_values, 'values'):
        shap_vals = np.abs(shap_values.values).mean(axis=0)
    else:
        shap_vals = np.abs(shap_values).mean(axis=0)

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'SHAP_Value': shap_vals
}).sort_values(by='SHAP_Value', ascending=False)

# Get top 12 features
top_12_features = feature_importance_df.head(12)

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(range(len(top_12_features)), top_12_features['SHAP_Value'].values, 
               color='steelblue', edgecolor='navy', linewidth=1.2)

# Customize y-axis labels
ax.set_yticks(range(len(top_12_features)))
ax.set_yticklabels(top_12_features['Feature'].values, fontsize=11)

# Add value labels on bars
for i, (idx, row) in enumerate(top_12_features.iterrows()):
    ax.text(row['SHAP_Value'] + 0.01, i, f"{row['SHAP_Value']:.4f}", 
            va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Mean |SHAP value|', fontsize=12, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=12, fontweight='bold')
ax.set_title('Figure    : Top 12 Feature Importance (SHAP Values)\nHorizontal bar chart with X-axis: mean |SHAP value|; Y-axis: feature name', 
             fontsize=13, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x', linestyle=':', linewidth=0.8)
ax.invert_yaxis()  # Highest importance at the top

plt.tight_layout()
plt.savefig(CONFIG.artifacts_dir / 'Figure_634_4_Feature_Importance_SHAP.png', dpi=300, bbox_inches='tight')
print("✅ Figure     saved to: Figure_634_4_Feature_Importance_SHAP.png")
plt.show()

# Print feature importance summary
print("\n" + "-"*80)
print("TOP 12 FEATURES BY MEAN |SHAP VALUE|")
print("-"*80)
print(top_12_features.to_string(index=False))

# Update metadata with feature importance
metadata['top_features_shap'] = top_12_features[['Feature', 'SHAP_Value']].to_dict('records')
joblib.dump(metadata, CONFIG.production_metadata_path)

print("\n" + "="*80)
print("✅ ALL FIGURES GENERATED SUCCESSFULLY")
print("="*80)
print(f"✓ Figure : Confusion Matrix - {CONFIG.artifacts_dir / 'Figure_634_1_Confusion_Matrix_XGBoost.png'}")
print(f"✓ Figure : ROC Curve - {CONFIG.artifacts_dir / 'Figure_634_2_ROC_Curve_MultiModel.png'}")
print(f"✓ Figure : Precision-Recall Curve - {CONFIG.artifacts_dir / 'Figure_634_3_PrecisionRecall_Curve.png'}")
print(f"✓ Figure : Feature Importance (SHAP) - {CONFIG.artifacts_dir / 'Figure_634_4_Feature_Importance_SHAP.png'}")
print("="*80)



FIGURE : TOP 12 FEATURE IMPORTANCE (SHAP VALUES)
Calculating SHAP values (this may take a moment)...
TreeExplainer failed (could not convert string to float: '[3.445227E2]'), trying PermutationExplainer...


PermutationExplainer explainer: 225it [00:38,  5.21it/s]                         


✅ Figure     saved to: Figure_634_4_Feature_Importance_SHAP.png

--------------------------------------------------------------------------------
TOP 12 FEATURES BY MEAN |SHAP VALUE|
--------------------------------------------------------------------------------
                                        Feature  SHAP_Value
                       dept_car_production_pune  101.331304
leave_type_daily_special_leave_not_call_on_duty   49.847444
                           department_avg_leave   49.027973
                        dept_quality_management    9.450979
                  leave_type_daily_casual_leave    4.632567
        leave_type_daily_earned_previlage_leave    4.291495
                      leave_type_daily_comp_off    3.602341
                     department_leave_frequency    2.713290
                               dept_hr_planning    2.582048
                                    leave_lag_7    2.346587
                    leave_type_daily_sick_leave    1.707376
                